# [RAG (Retrieval Augmented Generation)](https://docs.langchain.com/oss/python/deepagents/retrieval):

Large Language Models (LLMs) are powerful, but they have two key limitations:
- Finite context: they can’t ingest entire corpora at once.
- Static knowledge: their training data is frozen at a point in time.

- Retrieval addresses these problems by fetching relevant external knowledge at query time. This is the foundation of Retrieval-Augmented Generation (RAG), enhancing an LLM’s answers with context-specific information.


Core Phases of LangChain RAG

- Indexing Pipeline: Ingests, cleans, and splits your source documents into smaller manageable pieces (chunks).
- Embedding & Vector Storage: Transforms text chunks into numerical vectors using embedding models and stores them inside a vector database (such as FAISS, Pinecone, or Chroma) for fast semantic search.
- Retrieval Phase: Takes a user query, converts it into a vector, and searches the vector store to fetch the top-K most relevant document chunks.
- Generation Phase: Combines the original query and the retrieved context into a structured prompt template, then sends it to an LLM to formulate a grounded response.

### Retrieval Pipeline:

```mermaid

%%{init: {'flowchart': {'nodeSpacing': 12, 'rankSpacing': 18, 'padding': 4}, 'themeVariables': {'fontSize': '12px'}}}%%
flowchart TB
  subgraph ingest[" "]
    direction LR
    S(["Sources<br>(Google Drive, Slack, Notion, etc.)"]) --> L[Document Loaders]
    L --> A([Documents])
  end
  A --> B[Split into chunks]
  B --> C[Turn into embeddings]
  C --> D[(Vector Store)]
  Q([User Query]) --> E[Query embedding]
  E --> D
  D --> F[Retriever]
  F --> G[LLM uses retrieved info]
  G --> H([Answer])

  classDef trigger fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
  classDef process fill:#E5F4FF,stroke:#006DDD,stroke-width:2px,color:#030710
  classDef output fill:#EBD0F0,stroke:#885270,stroke-width:2px,color:#441E33
  classDef neutral fill:#F2FAFF,stroke:#40668D,stroke-width:2px,color:#2F4B68

  class S,Q trigger
  class L,B,C,E,F,G process
  class D output
  class A,H neutral
```

## RAG Architectures

Retrieval-Augmented Generation (RAG) can be implemented in multiple ways depending on the application's requirements for control, flexibility, latency, and reasoning.

The three common architectures are:

1. **2-Step RAG** – Retrieval always happens before generation.
2. **Agentic RAG** – An LLM-powered agent decides when and how to retrieve information.
3. **Hybrid RAG** – Combines characteristics of both approaches with validation and refinement steps.

### Architecture Comparison

| Architecture | Description | Control | Flexibility | Latency | Example Use Case |
|---|---|---|---|---|---|
| 2-Step RAG | Retrieval always happens before generation. Simple and predictable. | High | Low | Fast | FAQs, documentation bots |
| Agentic RAG | An LLM-powered agent decides when and how to retrieve during reasoning. | Low | High | Variable | Research assistants with access to multiple tools |
| Hybrid RAG | Combines characteristics of both approaches with validation steps. | Medium | Medium | Variable | Domain-specific Q&A with quality validation |

**Latency:** Latency is generally more predictable in 2-Step RAG because the maximum number of LLM calls is known and capped. Real-world latency can still vary depending on retrieval performance, API response times, network delays, and database queries.

---

### 1. 2-Step RAG

In 2-Step RAG, the retrieval step is always executed before the generation step.

The architecture is straightforward and predictable, making it suitable for applications where retrieving relevant documents is a clear prerequisite for generating an answer.

#### Workflow

1. User submits a question.
2. The system retrieves relevant documents from the knowledge base.
3. Retrieved documents are passed as context to the LLM.
4. The LLM generates an answer.
5. The answer is returned to the user.

#### Mermaid Diagram

```mermaid
%%{init: {'flowchart': {'nodeSpacing': 12, 'rankSpacing': 18, 'padding': 4}, 'themeVariables': {'fontSize': '12px'}}}%%
graph TB
    A[User Question] --> B["Retrieve Relevant Documents"]
    B --> C["Generate Answer"]
    C --> D[Return Answer to User]

    %% Styling
    classDef startend fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef process fill:#E5F4FF,stroke:#006DDD,stroke-width:1.5px,color:#030710

    class A,D startend
    class B,C process
```

#### Characteristics

- Retrieval always happens before generation.
- Simple execution flow.
- Predictable number of LLM calls.
- Less flexible because retrieval is mandatory.
- Suitable for FAQs and documentation-based question answering.

---

### 2. Agentic RAG

Agentic Retrieval-Augmented Generation combines the strengths of RAG with agent-based reasoning.

Instead of retrieving documents before answering, an agent powered by an LLM reasons about the user's question and decides when and how to retrieve information during the interaction.

The agent can choose whether external information is needed and which retrieval tools to invoke.

The only thing an agent needs to enable RAG behavior is access to one or more tools that can fetch external knowledge, such as:

- Documentation loaders
- Web search APIs
- Database queries
- Internal knowledge-base search

### Workflow

1. User submits a question.
2. The agent analyzes the question.
3. The agent decides whether external information is required.
4. If required, it invokes one or more retrieval tools.
5. The agent evaluates whether the retrieved information is sufficient.
6. If insufficient, it can perform additional retrieval.
7. Once sufficient information is available, the agent generates the final answer.

#### Mermaid Diagram

```mermaid
%%{init: {'flowchart': {'nodeSpacing': 12, 'rankSpacing': 18, 'padding': 4}, 'themeVariables': {'fontSize': '12px'}}}%%
graph TB
    A[User Input / Question] --> B["Agent (LLM)"]
    B --> C{Need external info?}
    C -- Yes --> D["Search using tool(s)"]
    D --> H{Enough to answer?}
    H -- No --> B
    H -- Yes --> I[Generate final answer]
    C -- No --> I
    I --> J[Return to user]

    %% Dark-mode friendly styling
    classDef startend fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef decision fill:#FDF3FF,stroke:#7E65AE,stroke-width:2px,color:#504B5F
    classDef process fill:#E5F4FF,stroke:#006DDD,stroke-width:1.5px,color:#030710

    class A,J startend
    class B,D,I process
    class C,H decision
```

#### Characteristics

- The agent dynamically decides whether retrieval is necessary.
- Supports multiple retrieval tools and data sources.
- Can perform iterative retrieval.
- More flexible than 2-Step RAG.
- Latency is variable because the agent may perform multiple reasoning and retrieval steps.
- Suitable for research assistants and complex question-answering systems.

---

### 3. Hybrid RAG

Hybrid RAG combines characteristics of both 2-Step and Agentic RAG.

It introduces intermediate steps such as query preprocessing, retrieval validation, and post-generation checks.

These systems offer more flexibility than fixed pipelines while maintaining some control over execution.

#### Typical Components

**1. Query Enhancement**

Modify the input question to improve retrieval quality. This can involve:

- Rewriting unclear queries.
- Generating multiple query variations.
- Expanding queries with additional context.

**2. Retrieval Validation**

Evaluate whether retrieved documents are relevant and sufficient.

If not, the system can refine the query and retrieve again.

**3. Answer Validation**

Check the generated answer for:

- Accuracy.
- Completeness.
- Alignment with source content.

If needed, the system can regenerate or revise the answer.

#### Workflow

1. User submits a question.
2. Query enhancement improves the question.
3. Documents are retrieved.
4. The system validates whether the retrieved information is sufficient.
5. If insufficient, the query is refined and retrieval is repeated.
6. If sufficient, the LLM generates an answer.
7. The generated answer is validated.
8. If the answer quality is poor, the system can try another approach.
9. The best answer is returned to the user.

#### Mermaid Diagram

```mermaid
%%{init: {'flowchart': {'nodeSpacing': 12, 'rankSpacing': 18, 'padding': 4}, 'themeVariables': {'fontSize': '12px'}}}%%
graph TB
    A[User Question] --> B[Query Enhancement]
    B --> C[Retrieve Documents]
    C --> D{Sufficient Info?}
    D -- No --> E[Refine Query]
    E --> C
    D -- Yes --> F[Generate Answer]
    F --> G{Answer Quality OK?}
    G -- No --> H{Try Different Approach?}
    H -- Yes --> E
    H -- No --> I[Return Best Answer]
    G -- Yes --> I
    I --> J[Return to User]

    classDef startend fill:#F6FFDB,stroke:#6E8900,stroke-width:2px,color:#2E3900
    classDef decision fill:#FDF3FF,stroke:#7E65AE,stroke-width:2px,color:#504B5F
    classDef process fill:#E5F4FF,stroke:#006DDD,stroke-width:1.5px,color:#030710

    class A,J startend
    class B,C,E,F,I process
    class D,G,H decision
```

#### Characteristics

- Combines fixed retrieval steps with conditional execution.
- Supports query rewriting and refinement.
- Validates retrieved documents before generation.
- Validates generated answers.
- Supports multiple iterations between retrieval and generation.
- Suitable for applications with ambiguous queries, quality requirements, and multiple information sources.

In [ ]:
import asyncio
import re
import json
import os
from pathlib import Path
from fnmatch import fnmatch
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from pydantic import BaseModel, Field, EmailStr, field_validator
from dotenv import load_dotenv
from os import getenv
from enum import Enum
from typing import List, Optional, Literal
import pandas as pd
import argparse
import logging
import uuid
from datetime import datetime, timezone
from threading import RLock
from typing import Annotated
import subprocess
import sys

# Lang chain
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents.middleware import AgentMiddleware, ToolCallRequest
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langchain_mcp_adapters.client import MultiServerMCPClient

from fastmcp import FastMCP, Client
from fastmcp.exceptions import ToolError

load_dotenv()
model = "openrouter/free"

llm = ChatOpenRouter(
    model_name=model,
)